# Tracking Open Models in AI Media · Stage 3: Implementing an RAG System for Question Answering

## Objective
Stage 3 builds and evaluates a question-answering system (RAG) over the open-models articles of the AI Media Dataset, in three steps:
1. **Text-only RAG**: a baseline that retrieves passages from the articles and generates answers grounded in these passages, with their publication dates.
2. **RAG with the knowledge graph**: the Stage 1 knowledge graph is added to the retrieval, so that answers can also use the entities and relations of the track; both systems are evaluated on the same test set and compared.
3. **Enhancement**: one limitation observed in the evaluation is addressed with one enhancement, whose effect is evaluated in turn.

The evaluation relies on a reviewed Q&A test set of the open-models track, built at the start of the stage.

## Link with the other stages
- **Input from Stage 1**: the cleaned corpus and the core open-models articles (the RAG corpus and the source of the test set), the WP1 lexicon (selection of track passages), the topics (coverage of the test set) and the knowledge graph with its schema (graph-augmented retrieval).

## Stage 3 requirements and where they are addressed
| Requirement (project brief) | Section |
|---|---|
| Select 100–150 representative passages of the track | 1.1–1.4 |
| Generate 200–300 Q&A pairs with an LLM different from the RAG model, mixing five question types | 1.5 |
| Manually review and refine the Q&A pairs | 1.6–1.7 |
| Set up a text-based RAG baseline from an existing implementation | 2 |
| Experiment with dense, sparse and hybrid retrieval | 2.4 |
| Ground answers in the retrieved passages, with publication dates for temporal questions | 2.5 |
| Integrate the Stage 1 knowledge graph and compare with text-only RAG | x |
| Evaluate faithfulness, relevance, context and temporal grounding | x |
| Implement and evaluate one RAG enhancement | x |

## Contributions
The work was split into three consecutive parts, each building on the previous one.

|Team member|	Sections|	Contribution|
|--|--|--|
|Jennifer Spieser |1, 2|Q&A Dataset Construction, text-based RAG |
|Jihyeon Choung |x|x|
|Daliasny Gonzalez Guzman |x| x|

# 1. Q&A Dataset Construction
To know whether the RAG system works, it must be tested on questions whose answer and source are known in advance. This first part builds that test set: question/answer pairs, each linked to the passage and the article it comes from. It is used to evaluate both the retrieval (is the source article found?) and the answers (are they correct and faithful to the sources?). To be useful, it must be representative (covering the subtopics, entities and periods of the track) and reliable (every question is clear and answerable from its source).

**Pipeline**
1. **Track articles**: the core open-models articles of Stage 1
2. **Passages**: the articles are cut into passages of a few paragraphs
3. **Track passages**: only passages about open models are kept
4. **Selection**: 150 passages covering periods, subtopics and entities
5. **Generation**: two questions per passage drafted by an LLM
6. **Manual review**: every pair is kept, edited or dropped
7. **Final test set**: the reviewed pairs are saved for the RAG system

**Cache**: every step saves its result in `QA_PATH` (shared Google Drive) and loads it at later runs instead of recomputing it. This matters most for the passage selection, which must never change once the questions are written from it.

## Setup
The notebook runs in Google Colab. It reads the Stage 1 outputs from the shared Google Drive and the WP1 lexicon from the project repository.

In [2]:
# anatoolbox
%pip install -q "anatoolbox[embeddings] @ git+https://github.com/umagira/anatoolbox"

from pathlib import Path
import json
import re
import subprocess
import time

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 120)
SEED = 42

# Clone the project repository
REPO = Path("/content/computational-language-technology-hs26")
if REPO.exists():
    !git -C {REPO} pull -q
else:
    !git clone -q https://github.com/jihyeon26/computational-language-technology-hs26.git {REPO}

# Mount Google Drive (input data and outputs)
from google.colab import drive
drive.mount("/content/drive")
# Inputs: Stage 1 outputs
DATA_DIR = Path("/content/drive/MyDrive/HSLU/CLT/data/processed")

OUT_DIR = Path("/content/drive/MyDrive/HSLU/CLT/extraction_outputs/stage3/wp1") # wp1 outputs
QA_PATH = OUT_DIR / "qa_dataset" # Q&A test set
RAG_DIR = Path("/content/drive/MyDrive/HSLU/CLT/extraction_outputs/stage3/wp1/rag") # RAG outputs

for directory in (QA_PATH, RAG_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# WP1 lexicon of Stage 1
lexicon = json.loads((REPO / "artifacts/stage1_wp1/lexicon/open_models_lexicon.json").read_text(encoding="utf-8"))
FAMILY_RX = {f["label"]: re.compile(f["rx"]) for f in lexicon["families"].values()}
CONCEPT_RX = {name: re.compile(c["rx"]) for name, c in lexicon["concepts"].items()}

# Matching rules of Stage 1 (stage1_final.ipynb): special dashes and quotes are replaced before matching,
# and "Llama" inside another product name (Llama Nemotron, Llama Stack, LlamaCon) is not the Llama family
MATCH_MAP = str.maketrans({"‐": "-", "‑": "-", "‒": "-", "–": "-", "−": "-", "‘": "'", "’": "'", "“": '"', "”": '"'})
FAMILY_EXCEPTIONS = {"Llama": re.compile(r"\bLlama[\w.\- ]{0,15}?(?:Nemotron|Stack)\b|\bLlamaCon\b")}

def family_view(name, text):
    # Text in which a family is searched: matching view, without the product names that contain the family name
    view = text.translate(MATCH_MAP)
    return FAMILY_EXCEPTIONS[name].sub(" ", view) if name in FAMILY_EXCEPTIONS else view

# File names of Q&A
PASSAGES_PATH = QA_PATH / "passages_for_qa.csv"
DRAFT_PATH = QA_PATH / "qa_draft.csv"
REVIEWED_PATH = QA_PATH / "qa_reviewed.csv"
FINAL_JSON = QA_PATH / "qa_reviewed.json"



  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Mounted at /content/drive


## 1.1 Track Articles
Stage 1 gave every article a relevance tier for the open-models track. The test set uses the core articles (`om_tier == "core"`), the same collection as the RAG corpus, so that every question then comes from an article the RAG can retrieve. The relevance tiers of Stage 1 are:
| Tier | Meaning | Used here |
| --| --| --|
| `core`| article is about open models | yes|
| `mention`| article mainly deals wih another topic and only names open models| no: a question drawn from it would rarely be about the track|
|`reference`| model documentation pages (API reference), not media articles| no: already excluded in Stage 2|
| `none`| no open-model term | no |

In addition, the Stage 1 knowledge graph was built from the core articles. The text-only RAG uses the same articles, so that it can be compared with the graph-enhanced RAG on the same evidence.

Two files are combined:
- `corpus_clean.parquet`: the cleaned text, publication date, quarter, source domain and the identifiers of every articles;
- `open_models_core.parquet`: the dominant topic of each core article (Stage 1 topic model), user here as the subtopic of the article.

Each article keeps two identifiers: `doc_id`, the identifier used in previous stages, and `src_row`, its row number.

In [3]:
# Core articles
corpus = pd.read_parquet(DATA_DIR / "corpus_clean.parquet", columns=["doc_id","src_row","date","quarter","domain","title","text","om_tier"])
articles = corpus[corpus["om_tier"] == "core"].drop(columns="om_tier")

# Subtopic of each article
topics = pd.read_parquet(DATA_DIR / "open_models_core.parquet", columns=["doc_id","dominant_topic_label"])
articles = articles.merge(topics, on="doc_id", how="left").rename(columns={"dominant_topic_label":"topic"})
articles['date'] = pd.to_datetime(articles["date"]).dt.strftime("%Y-%m-%d")

print(f"Core articles: {len(articles):,} | topics: {articles["topic"].nunique()}"
f" | without topic: {articles["topic"].isna().sum()}")
display(articles["quarter"].value_counts().sort_index().rename("articles").to_frame().T)

Core articles: 1,210 | topics: 16 | without topic: 0


quarter,2024Q3,2024Q4,2025Q1,2025Q2,2025Q3,2025Q4,2026Q1,2026Q2,2026Q3
articles,22,104,349,156,103,90,99,179,108


The 1,210 core articles cover 16 topics. They are not distributed evenly across the quarters: they peak in 2025Q1, followed by 2025Q2 and the number then decreases until 2026Q1 and rises again in 2026Q2. Because of this uneven distribution, a random sample of passages would be dominated by 2025Q1, which is why the passages will be selected with an equal share per quarter.

## 1.2 Passages
A question is written from a passage of one or a few consecutive paragraphs, not from a whole article: long enough to contain a fact with its context, short enough to check during the review, and close in size to the chunks the RAG system retrieves.

**How the articles are cut**: whole paragraphs are joined until a passage has more than `MIN_WORDS` (120) words, without going over `MAX_WORDS` (250); a paragraph longer than `MAX_WORDS` is first cut at sentence boundaries. Pieces that stay too short and single sentences longer than `MAX_WORDS` are dropped.

Each passage gets an identifier `passage_id` = `doc_id` + its position (`_p00`, `_p01`, …), so it can always be traced back to its article.

In [4]:
# Size of a passage
MIN_WORDS = 120
MAX_WORDS = 250

# End of a sentence
SENT_SPLIT = re.compile(r"(?<=[.!?])\s+(?=[A-Z0-9\"'(])")

def n_words(text):
  return len(text.split())

def units(text):
  # Paragraph of an article
  for paragraph in (p.strip() for p in text.split("\n")):
    if not paragraph:
      continue
    if n_words(paragraph) <= MAX_WORDS:
      yield paragraph
    else:
      yield from (s.strip() for s in SENT_SPLIT.split(paragraph) if s.strip())

def split_passages(text):
  # Join consecutive units until passage has min_words but less than max_words
  passages = []
  current = []
  for unit in units(text):
    if current and n_words(" ".join(current + [unit])) > MAX_WORDS:
      passages.append(current)
      current = []
    current.append(unit)
    if n_words(" ".join(current)) > MIN_WORDS:
      passages.append(current)
      current = []
  # Drop short and long pieces
  passages = [p for p in passages if MIN_WORDS <= n_words(" ".join(p)) <= MAX_WORDS]
  return ["\n\n".join(p) for p in passages]

# All passages of all core articles, with the information of their article
rows = []
for art in articles.itertuples(index=False):
    for i, passage in enumerate(split_passages(art.text)):
        rows.append({"passage_id": f"{art.doc_id}_p{i:02d}", "doc_id": art.doc_id, "src_row": art.src_row,
                     "date": art.date, "quarter": art.quarter, "domain": art.domain, "title": art.title,
                     "topic": art.topic, "position": i, "n_words": n_words(passage), "text": passage})
passages = pd.DataFrame(rows)

print(f"Passages: {len(passages):,} from {passages['doc_id'].nunique():,} articles "
      f"(median {passages.groupby('doc_id').size().median():.0f} per article)")
display(passages["n_words"].describe().round(0).to_frame("words per passage").T)

Passages: 12,767 from 1,210 articles (median 8 per article)


,count,mean,std,min,25%,50%,75%,max
words per passage,12767.0,150.0,25.0,120.0,131.0,144.0,162.0,250.0


In [5]:
def count_families(text):
    # Open-model families found in the text
    counts = {name: sum(1 for _ in rx.finditer(family_view(name,text))) for name, rx in FAMILY_RX.items()}
    return ", ".join(f"{name} ({n})" for name, n in counts.items() if n) or "-"

# Every passage of one example article
example = passages[passages["doc_id"] == passages["doc_id"].iloc[0]]
print(f"Example: '{example['title'].iloc[0]}' -> {len(example)} passages")
display(pd.DataFrame({
    "passage": example["passage_id"],
    "words": example["n_words"],
    "paragraphs": example["text"].str.count("\n\n") + 1,
    "open-model families (occurrences)": [count_families(t) for t in example["text"]],
    "start of the passage": example["text"].str[:80] + "...",
}).reset_index(drop=True))

Example: 'Deploying your Llama Model via vLLM using SageMaker Endpoint' -> 6 passages


,passage,words,paragraphs,open-model families (occurrences),start of the passage
0,doc_c44fd663ac_p00,144,3,Llama (1),"In any machine learning project, the goal is to train a model that can be used b..."
1,doc_c44fd663ac_p01,130,4,-,DJL (Deep Java Library) is an open-source library developed by AWS used to devel...
2,doc_c44fd663ac_p02,168,3,-,The three earlier stated components are viewable on SageMaker’s page and they ea...
3,doc_c44fd663ac_p03,137,4,-,"Optionally, you can also define the subnets, and security group of the managed i..."
4,doc_c44fd663ac_p04,138,4,-,You may locate the URL in the AWS Management Console > SageMaker > Endpoints > [...
5,doc_c44fd663ac_p05,138,3,-,"Note that to use chat completion, the request payload must be within the message..."


The example article is cut into 6 passages of 130 to 168 words, each made of 3 or 4 paragraphs. Although the article is about deploying a Llama model, Llama appears only once. This shows that a passage of an article is therefore not necessarily about the track: drawing questions from all passages would bring off-track question into the test set, which is why the passages are filtered in the next section.

## 1.6 Track passages
A passage is kept only if it mentions an open-model family or an openness concepts of the WP1 lexicon of Stage 1, using the same regular expression:
- **families** (e.g. Llama, Qwen, Mistral, DeepSeek, Gemma): the patterns also recognise version names such as *Llama 3.1* or *Qwen3*;
- **openness concepts** (e.g. *open-weight*, *open-source model*): the pattern for open source only counts it when it refers to a model (*open-source model*, *open-source LLM*), not to software or a community.



In [6]:
def find_families(text):
    # Open-model families mentioned in a text - stage 1 matching rules
    return [name for name, rx in FAMILY_RX.items() if rx.search(family_view(name, text))]

def find_concepts(text):
    # Openness concepts mentioned in a text - matching view of stage 1
    return [name for name, rx in CONCEPT_RX.items() if rx.search(text.translate(MATCH_MAP))]

passages["families"] = [find_families(t) for t in passages["text"]]
passages["concepts"] = [find_concepts(t) for t in passages["text"]]
passages["on_track"] = passages["families"].str.len().gt(0) | passages["concepts"].str.len().gt(0)

track_passages = passages[passages["on_track"]].reset_index(drop=True)
print(f"Track passages: {len(track_passages):,} of {len(passages):,} ({passages['on_track'].mean():.0%}) "
      f"from {track_passages['doc_id'].nunique():,} articles")
print(f"Families found: {track_passages['families'].explode().nunique()} | "
      f"passages with a family: {track_passages['families'].str.len().gt(0).mean():.0%}")
display(track_passages["families"].explode().value_counts().head(15).rename("passages").to_frame().T)

Track passages: 5,988 of 12,767 (47%) from 1,205 articles
Families found: 35 | passages with a family: 86%


families,DeepSeek,Llama,Qwen,Gemma,Mistral,Kimi,Nemotron,GLM,Phi,gpt-oss,Stable Diffusion / StableLM,MiniMax,OLMo / Molmo / Tülu,Granite,FLUX
passages,2023,1004,733,688,484,331,325,199,135,131,119,107,63,58,54


Less than half of the passages (47%) mention an open-model family or an openness concept. The track passages mention 35 different families, dominated by DeepSeek and Llama; 86% of them mention at least one family, while the other 14% are kept for an openness concept only.

As for the quarters, the families are unevenly distributed: a random sample of passages would be dominated by DeepSeek and Llama, and small families might not appear at all. The passages are therefore selected by rules that spread them over periods, subtopics and families in the next section.

## 1.4 Selection of 150 passages
The 150 passages are selected by rules, so that every period, every subtopic and many families are covered:
| Rule | How it works |
|---|---|
| **Periods** | the passages are shared equally between the quarters; a quarter with too few articles gives all it has, and the rest goes to the others |
| **Subtopics** | within a quarter, the next passage comes from the topic with the fewest passages selected so far |
| **Families** | within that topic, a passage mentioning a family not yet covered is preferred |
| **One passage per article** | avoids closely related questions and covers more articles |
| **Fixed seed** | the candidates are shuffled once, so the selection is the same at every run |

The selection is saved in `passages_for_qa.csv`and loaded at every later run.

In [7]:
N_PASSAGES = 150

def allocate(total, available):
    # Equal share per quarter; a quarter with fewer candidates gives all it has, the rest goes to the others
    share, left, todo = {}, total, dict(available)
    while todo:
        equal = left // len(todo)
        small = {q: n for q, n in todo.items() if n <= equal}
        if not small:                                        # every remaining quarter can give its share
            for i, q in enumerate(sorted(todo)):
                share[q] = equal + (1 if i < left - equal * len(todo) else 0)
            break
        for q, n in small.items():                           # quarters with too few articles give all they have
            share[q] = n
            left -= n
            del todo[q]
    return share

def select(candidates, n_total):
    # Select n_total passages: equal share per quarter, topics in turn, new model families first
    pool = candidates.sample(frac=1, random_state=SEED)
    share = allocate(n_total, pool.groupby("quarter")["doc_id"].nunique().to_dict())
    topic_count = {t: 0 for t in pool["topic"].unique()}                  # passages per topic in the selection
    covered_families, used_docs, chosen = set(), set(), []
    for quarter in sorted(share):
        quarter_pool = pool[pool["quarter"] == quarter]
        for _ in range(share[quarter]):
            quarter_pool = quarter_pool[~quarter_pool["doc_id"].isin(used_docs)]   # one passage per article
            if quarter_pool.empty:
                break
            # 1. the topic of this quarter with the fewest passages in the selection so far
            topic = min(quarter_pool["topic"].unique(), key=lambda t: (topic_count[t], t))
            in_topic = quarter_pool[quarter_pool["topic"] == topic]
            # 2. within it, the first passage that mentions a family not covered yet (else the first one)
            new_family = in_topic["families"].map(lambda fams: bool(set(fams) - covered_families))
            row = in_topic[new_family].iloc[0] if new_family.any() else in_topic.iloc[0]
            chosen.append(row)
            topic_count[topic] += 1
            covered_families.update(row["families"])
            used_docs.add(row["doc_id"])
    return pd.DataFrame(chosen).sort_values(["date", "passage_id"]).reset_index(drop=True)

# Selection created once, then always loaded
if PASSAGES_PATH.exists():
    selected = pd.read_csv(PASSAGES_PATH, converters={"families": json.loads, "concepts": json.loads})
    print(f"Selection loaded: {len(selected)} passages ({PASSAGES_PATH.name})")
else:
    selected = select(track_passages, N_PASSAGES)
    to_save = selected.drop(columns=["on_track"]).assign(families=selected["families"].map(json.dumps),
                                                        concepts=selected["concepts"].map(json.dumps))
    to_save.to_csv(PASSAGES_PATH, index=False)
    print(f"Selection created and saved: {len(selected)} passages ({PASSAGES_PATH.name})")

Selection loaded: 150 passages (passages_for_qa.csv)


In [8]:
# Periods
by_quarter = pd.DataFrame({"available articles": track_passages.groupby("quarter")["doc_id"].nunique(),
                           "selected passages": selected["quarter"].value_counts()}).fillna(0).astype(int)
display(by_quarter.T)

quarter,2024Q3,2024Q4,2025Q1,2025Q2,2025Q3,2025Q4,2026Q1,2026Q2,2026Q3
available articles,22,103,349,152,103,90,99,179,108
selected passages,17,17,17,17,17,17,16,16,16


The passages are balanced across the quarters; between 16 and 17 passages.

In [9]:
# Subtopics
by_topic = pd.DataFrame({"available articles": track_passages.groupby("topic")["doc_id"].nunique(),
                         "selected passages": selected["topic"].value_counts()}).fillna(0).astype(int)
display(by_topic.sort_values("selected passages", ascending=False))

,available articles,selected passages
topic,,
T04 claude · agent · cod,126,10
T03 blackwell · gpu · workload,52,10
T07 china · united_states · chinese,81,10
T05 memory · attention · token,75,10
T08 nemotron · nvidia · nvidia nemotron,63,10
T10 api · file · python,105,10
T11 english · multilingual · instruction,44,10
T02 deepseek · r1 · chinese,103,9
T06 image · video · text,58,9


Every topic receives between 8 and 10 passages.

In [10]:
# Model families
all_families = set(track_passages["families"].explode().dropna())
sel_families = selected["families"].explode().dropna()
print(f"Families covered: {sel_families.nunique()} of {len(all_families)} found in the track passages")

# Families not covered
missing = sorted(all_families - set(sel_families))
if missing:
    mentions = track_passages.explode("families")
    mentions = mentions[mentions["families"].isin(missing)]
    display(mentions.groupby("families").agg(passages=("passage_id", "size"), articles=("doc_id", "nunique")))
else:
    print("Families not covered: none")


Families covered: 34 of 35 found in the track passages


,passages,articles
families,,
Snowflake Arctic,1,1


The selection covers 34 of the 35 open-model families; only *Snowflake Arctic* is missing as it appears in a single passage from one article, which was not reached by the selection rules.

## 1.5 Generation of the Q&A pairs
The questions are drafted by Claude Opus 5.5 (Anthropic), used through Claude Code outside this notebook. The model received each selected passage (`passages_for_qa.csv`) with its title, date and identifiers, and returned `qa_draft.csv`, loaded below.

**Questions types** (two per passage, of two different types, each type about one fifth of the test set):
| Type | What it tests |
|---|---|
| **factual** | a single fact of the passage |
| **temporal** | a date, a period or an order of events |
| **relational** | a link between two entities (developer, partner, base model, competitor) |
| **analytical** | a reason, a cause or a consequence |
| **comparative** | a comparison of models, companies or results |

**Rules given to the model**:
- Write two questions, of two different types, chosen among the five types above according to what the passage actually contains
- Each question must be answerable from the passage alone (plus its publication date for temporal questions), without outside knowledge.
- Each question must be self-contained: it names the model, company or event it is about, and never refers to "the article", "the passage" or "this model", since a RAG user asks without seeing any source.
- The question must not contain its own answer.
- The reference answer is short (one or two sentences), complete and only uses facts from the passage.
- The evidence is a sentence copied word for word from the passage that supports the answer, so that the answer can be checked quickly during the review.
- Questions are phrased as a user would ask them: short, one question at a time, without referring to the source (no "according to the article" or "in a report"), and without copying the wording of the passage beyond the names needed to identify the model, company or event.

In [11]:
# Drafted questions
draft = pd.read_csv(DRAFT_PATH, dtype={"review_status": str, "review_note": str})

print(f"Draft loaded: {len(draft)} questions from {draft['passage_id'].nunique()} passages ({DRAFT_PATH.name})")

display(draft["type"].value_counts().rename("questions").to_frame().T)
display(draft[["qa_id", "type", "question", "reference_answer"]].head(6))

Draft loaded: 300 questions from 150 passages (qa_draft.csv)


type,factual,analytical,relational,comparative,temporal
questions,69,62,59,57,53


,qa_id,type,question,reference_answer
0,q001,factual,How can I deploy a Llama model with vLLM on AWS?,"AWS SageMaker endpoint, with its DJL (Deep Java Library) image."
1,q002,relational,Which company is behind SageMaker?,"AWS (Amazon Web Services): SageMaker is an AWS service for managing the machine learning lifecycle, and its inferenc..."
2,q003,temporal,When did Google launch DataGemma?,On 12 September 2024: an article published that day reports that Google debuted DataGemma that day.
3,q004,relational,Which model family is DataGemma built on?,Google's Gemma family of open models; DataGemma grounds its answers in data from Google's Data Commons platform.
4,q005,factual,How fast can SambaNova run Llama 3.1 405B?,More than 100 tokens per second; the 70B model runs at up to 580 tokens per second.
5,q006,comparative,How is SambaNova's way of running Llama different from Groq and Cerebras?,"SambaNova says Groq and Cerebras store model weights on SRAM, which requires many chips, whereas SambaNova runs all ..."


## 1.6 Manual review
As an LLM can misread a passage, write ambiguous question or give an answer that is not fully supported, every drafted question was manually reviewed by the team members against these criteria:

| Criterion | Question to ask |
|---|---|
| **Clarity** | Is the question understandable without the passage, with one clear answer? |
| **Correctness** | Is the reference answer true according to the passage, and complete? |
| **Answerability** | Can the answer be found in the passage (or its date), without outside knowledge? |
| **Diversity** | Does the question ask a different fact from the other questions? |
| **Coverage** | Is the question about the open-models track, and is its type right? |
| **Naturalness** | Would a user ask it without having read the article? |

Each question is marked `keep`, `edit` (corrected in the row) or `drop` (off-track, too general, duplicate or not answerable), with a `review_note` for every `edit` and `drop`; the result is saved as `qa_reviewed.csv`.

### 1.6.1 Complementary temporal questions
After the first review, temporal questions were the smallest group (28 of 218), and most dates written in the 150 passages were already used. Fifteen extra passages with an explicit date are therefore selected below, with the same rules as the first selection, among the track passages that come from unused articles and contain a sentence with both an explicit date (e.g. *May 2024*, *in 2023*) and an open-model family or openness concept. Thirteen temporal questions were drafted by Claude Opus 5.5 from their dated sentences and reviewed with the same criteria.

In [12]:
# Explicit date
DATE_RX = re.compile(r"\b(?:January|February|March|April|May|June|July|August|September|October|November|December"
                     r"|Jan\.|Feb\.|Mar\.|Apr\.|Aug\.|Sept?\.|Oct\.|Nov\.|Dec\.)\s+(?:\d{1,2},?\s+)?(?:19|20)\d\d\b"
                     r"|\b(?:in|since|from|by|until|before|after|early|late|mid)[\s-]+(?:19|20)\d\d\b(?![\-/]\d)")
SENTENCE_RX = re.compile(r"[^.!?\n]+[.!?]?")

def dated_track_sentences(text):
    # Sentences that contain both an explicit date and an open-model family or openness concept
    return [s.strip() for s in SENTENCE_RX.findall(text)
            if DATE_RX.search(s) and (find_families(s) or find_concepts(s))]

N_EXTRA = 15
EXTRA_PATH = QA_PATH / "passages_temporal_extra.csv"

# Extra passages for temporal questions: created once then always loaded
if EXTRA_PATH.exists():
    extra = pd.read_csv(EXTRA_PATH, converters={"families": json.loads, "concepts": json.loads})
    print(f"Extra selection loaded: {len(extra)} passages ({EXTRA_PATH.name})")
else:
    # Candidates: track passages with a dated sentence about the track, from articles not used in the first selection
    candidates = track_passages[~track_passages["doc_id"].isin(selected["doc_id"])].copy()
    candidates["dated_sentences"] = [dated_track_sentences(t) for t in candidates["text"]]
    candidates = candidates[candidates["dated_sentences"].str.len() > 0]
    extra = select(candidates, N_EXTRA)                  # same rules as the first selection (quarters, topics, families)
    extra["dated_sentences"] = extra["dated_sentences"].map(" | ".join)
    extra.drop(columns=["on_track"], errors="ignore").assign(
        families=extra["families"].map(json.dumps), concepts=extra["concepts"].map(json.dumps)).to_csv(EXTRA_PATH, index=False)
    print(f"Extra selection created and saved: {len(extra)} passages from {len(candidates):,} candidates ({EXTRA_PATH.name})")

Extra selection loaded: 15 passages (passages_temporal_extra.csv)


## 1.7 Final test set
The reviewed file (`qa_reviewed.csv`) is loaded and checked, the dropped questions are removed, and the coverage of the final test set is shown by type, quarter, subtopic and family. The final test set is saved as `qa_reviewed.json` for the RAG part: each question keeps its `id`, `type`, `question`, `reference_answer`, `source_ids` (the `doc_id` of its article, as a list so that other articles can be added), `date`, `passage_id`, `quarter` and `topic`.

In [13]:
TYPES = ["factual", "temporal", "relational", "analytical", "comparative"]


reviewed = pd.read_csv(REVIEWED_PATH, dtype={"review_status": str, "review_note": str})
reviewed["review_status"] = reviewed["review_status"].str.strip().str.lower()

# Check that every row have been checked
bad_status = ~reviewed["review_status"].isin(["keep", "edit", "drop"])
bad_type = ~reviewed["type"].isin(TYPES)
assert not bad_status.any(), f"{bad_status.sum()} rows without a valid review_status (keep / edit / drop)"
assert not bad_type.any(), f"{bad_type.sum()} rows with an unknown type"

# Result of the check
print("Review:", reviewed["review_status"].value_counts().to_dict())

# Final test set
final = reviewed[reviewed["review_status"] != "drop"].reset_index(drop=True)
print(f"\nFinal test set: {len(final)} questions from {final['passage_id'].nunique()} passages")

Review: {'keep': 140, 'drop': 100, 'edit': 73}

Final test set: 213 questions from 135 passages


In [14]:
# Question types
display(pd.crosstab(final["type"], final["quarter"], margins=True, margins_name="total"))

# Subtopics and families of the questions (through their passage)
families = final["passage_id"].map(selected.set_index("passage_id")["families"])
print(f"Topics covered: {final['topic'].nunique()} of {articles['topic'].nunique()} | "
      f"families covered: {families.explode().nunique()} | articles: {final['doc_id'].nunique()}")
display(final["topic"].value_counts().rename("questions").to_frame().T)

quarter,2024Q3,2024Q4,2025Q1,2025Q2,2025Q3,2025Q4,2026Q1,2026Q2,2026Q3,total
type,,,,,,,,,,
analytical,2,3,2,4,3,4,3,4,2,27
comparative,7,4,3,4,4,5,5,6,5,43
factual,10,9,7,5,4,6,5,7,4,57
relational,4,6,6,4,5,5,9,4,5,48
temporal,3,4,6,4,4,4,4,4,5,38
total,26,26,24,21,20,24,26,25,21,213


Topics covered: 16 of 16 | families covered: 34 | articles: 135


topic,T05 memory · attention · token,T08 nemotron · nvidia · nvidia nemotron,T11 english · multilingual · instruction,T13 meta · zuckerberg · llama,T04 claude · agent · cod,T02 deepseek · r1 · chinese,T09 gemma · gemma model · model gemma,T03 blackwell · gpu · workload,T15 qwen · qwen qwen · context length,T01 reasoning · reinforcement · reinforcement learning,T16 people · ask · find,T07 china · united_states · chinese,T06 image · video · text,T10 api · file · python,T12 stock · investor · growth,T14 enterprise · innovation · solution
questions,19,19,18,15,15,15,14,13,13,13,11,11,10,10,10,7


The final test set contains 213 questions from 135 passages. All quarters are represented, as all 16 subtopics and 34 open-model families.

In [15]:
# Save the final test set
records = [{"id": r.qa_id, "type": r.type, "question": r.question, "reference_answer": r.reference_answer,
            "source_ids": [r.doc_id], "date": r.date, "passage_id": r.passage_id,
            "quarter": r.quarter, "topic": r.topic} for r in final.itertuples(index=False)]
FINAL_JSON.write_text(json.dumps(records, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved: {FINAL_JSON.name} ({len(records)} questions)")
display(pd.DataFrame(records).head(5))

Saved: qa_reviewed.json (213 questions)


,id,type,question,reference_answer,source_ids,date,passage_id,quarter,topic
0,q001,factual,Which AWS service can I use to deploy a Llama model with vLLM?,"AWS SageMaker endpoint, with its DJL (Deep Java Library) image.",[doc_c44fd663ac],2024-09-12,doc_c44fd663ac_p00,2024Q3,T10 api · file · python
1,q003,temporal,When did Google launch DataGemma?,On 12 September 2024: an article published that day reports that Google debuted DataGemma that day.,[doc_d57f4c755c],2024-09-12,doc_d57f4c755c_p00,2024Q3,T09 gemma · gemma model · model gemma
2,q004,relational,Which model family is DataGemma built on?,Google's Gemma family of open models; DataGemma grounds its answers in data from Google's Data Commons platform.,[doc_d57f4c755c],2024-09-12,doc_d57f4c755c_p00,2024Q3,T09 gemma · gemma model · model gemma
3,q005,factual,How fast can SambaNova run Llama 3.1 405B?,More than 100 tokens per second; the 70B model runs at up to 580 tokens per second.,[doc_3f5ee378c1],2024-09-13,doc_3f5ee378c1_p04,2024Q3,T03 blackwell · gpu · workload
4,q006,comparative,How is SambaNova's way of running Llama different from Groq and Cerebras?,"According to McGonnell, Groq and Cerebras store model weights on SRAM, which reequires many chips, whereas SambaNova...",[doc_3f5ee378c1],2024-09-13,doc_3f5ee378c1_p04,2024Q3,T03 blackwell · gpu · workload


## 1.8 Saved outputs
| File  | Content |
|---|---|
| `passages_for_qa.csv` | the 150 passages of the first selection, with their article, date, quarter, topic and families |
| `passages_temporal_extra.csv` | the 15 extra passages with an explicit date, with their dated sentences |
| `qa_draft.csv` | the questions drafted by the LLM before the review and temporal questions from the extra passages |
| `qa_reviewed.csv` | all drafted questions after the manual review |
| `qa_reviewed.json` | the final test set (kept and edited questions only) |

# 2. RAG Implementation
This part builds the text-only RAG baseline, following the skeleton ([`course_project_skeleton.ipynb`](https://github.com/umagira/anatoolbox/blob/main/notebooks/course_project_skeleton.ipynb) and its toolbox `anatoolbox`. The steps and tools of the skeleton are kept, so that the other parts of Stage 3 can reuse them unchanged; they are adapted to the track: the core articles of Stage 1 as corpus, the reviwed test set of section 1, and a local open model for generation.

**Pipeline**
1. **Language model**: an open model (Qwen3:4b-instruct) runs locally in Colab
2. **Corpus**: the 1,210 core open-models articles of Stage 1, the same articles as the Q&A test set
3. **Chunks and embeddings**: the articles are cut into overlapping chunks, indexed for keyword search and embedded for semantic search
4. **Retrieval strategies**: sparse, dense and hybrid search, compared on example questions of the test set
5. **Grounded generation**: the model answers only from the retrieved passages, cites them and uses their publication dates

Every long step (embeddings) is saved in `RAG_DIR` on Google Drive and loaded at later runs.

## Setup
The RAG system uses Qwen3-4B-Instruct-2507, an open model run locally in Colab with Ollama. The model is downloaded at each new Colab session.

*Note*: section 2 was run on a L4 GPU. To run it alone, run the setup cell at the top of the notebook first.

In [16]:
# Anatoolbox tools
import anatoolbox
from anatoolbox import ToolContext, resolve_tools

anatoolbox.register_reference_tools(replace=True)
ctx = ToolContext()
ingest, chunk, retrieve, synthesize = resolve_tools(
    ["ingest_corpus", "chunk_by_size", "retrieve_passages", "synthesize_answer"])
print("anatoolbox", anatoolbox.__version__)

# Install and start Ollama
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
ollama_server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

RAG_MODEL = "qwen3:4b-instruct"
!ollama pull {RAG_MODEL}

anatoolbox 0.1.0
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.



In [17]:
from anatoolbox.llm_client import call_llm_text, configure_llm, model_for

# Point every anatoolbox tool at the local Ollama server
configure_llm(base_url="http://localhost:11434/v1", model=RAG_MODEL, timeout=180, max_retries=2)

# Check that the model answers
started = time.perf_counter()
reply = call_llm_text("Reply with one word.", "Say: ready", model=model_for("default"), max_tokens=20)
print(f"{model_for('default')} answered {reply!r} in {time.perf_counter() - started:.1f} s")

qwen3:4b-instruct answered 'ready' in 84.7 s


## 2.1 Corpus
The RAG corpus is the 1,210 core open-models articles of Stage 1, the same collection as the test set (Section 1.1). Each article keeps its Stage 1 `doc_id` as `id`, so that a retrieved passage can be matched to the `source_ids` of the test set; its title, publication date, URL and domain are kept as metadata and shown to the language model with every passage.

In [18]:
# Load corpus
corpus = pd.read_parquet(DATA_DIR / "corpus_clean.parquet",
                         columns=["doc_id", "date", "domain", "url", "title", "text", "om_tier"])
core = corpus[corpus["om_tier"] == "core"]
core_csv = pd.DataFrame({"id": core["doc_id"], "title": core["title"], "text": core["text"],
                         "date": pd.to_datetime(core["date"]).dt.strftime("%Y-%m-%d"),
                         "url": core["url"], "domain": core["domain"]})
CORE_CSV = RAG_DIR / "core_articles.csv"
core_csv.to_csv(CORE_CSV, index=False)

# Load the articles into anatoolbox
rag_articles = ingest.run({"path": str(CORE_CSV), "name": "core_articles"}, context=ctx)
print(f"{rag_articles['records']:,} core articles, {core_csv['date'].min()} to {core_csv['date'].max()}")

1,210 core articles, 2024-09-12 to 2026-08-31


### 2.1.1 Test set
The reviewed Q&A test set (`qa_reviewed.json`) is loaed to choose example questions and to verify that every source article is in the corpus

In [19]:
# Load qa reviewed
EVAL_SET = json.loads((QA_PATH / "qa_reviewed.json").read_text(encoding="utf-8"))
corpus_ids = set(core_csv["id"])
missing = [q["id"] for q in EVAL_SET if not set(q["source_ids"]) <= corpus_ids]
print(f"{len(EVAL_SET)} test questions | source articles missing from the corpus: {len(missing)}")
display(pd.Series([q["type"] for q in EVAL_SET]).value_counts().rename("questions").to_frame().T)

213 test questions | source articles missing from the corpus: 0


,factual,relational,comparative,temporal,analytical
questions,57,48,43,38,27


## 2.2 Chunks
A question is usually answered by a few sentences, so the articles are cut into chunks of fixed size, the baseline method of the toolbox:
- **200 words**: enough for a fact with its context, while keeping several chunks within the prompt;
- **40 words of overlap**, so that a statement cut at a boundary appears whole in one of the two chunks;
- **the article title added to every chunk** (`contextualize="title"`): a chunk often does not name what it is about (*"the model"*), and the title restores that context, as in the review of the test set, where the title counted as part of the source.

The passages of section 1 are not reused: they only cover the track passages, while the RAG system must search the full text of every article.

In [20]:
# Parameters
CHUNK_ARGS = {"size": 200, "overlap": 40, "contextualize": "title"}

# Corpus
from anatoolbox.corpus import get_corpus

chunks = chunk.run({"input": rag_articles, **CHUNK_ARGS}, context=ctx)
print(f"{chunks['chunks']:,} chunks from {chunks['articles']:,} articles | words per chunk: {chunks['tokens']}")
display(pd.DataFrame(get_corpus(chunks["corpus"]).records[:3])[["id", "source_id", "title", "date", "chunk_text"]])

12,720 chunks from 1,210 articles | words per chunk: {'min': 41, 'median': 200.0, 'max': 200}


,id,source_id,title,date,chunk_text
0,doc_c44fd663ac#0,doc_c44fd663ac,Deploying your Llama Model via vLLM using SageMaker Endpoint,2024-09-12,"In any machine learning project, the goal is to train a model that can be used by others to derive a good prediction..."
1,doc_c44fd663ac#1,doc_c44fd663ac,Deploying your Llama Model via vLLM using SageMaker Endpoint,2024-09-12,"docker images, including vLLM [ 2 ]. This image is used in the SageMaker endpoint.\n\nvLLM, introduced in 2023, is a..."
2,doc_c44fd663ac#2,doc_c44fd663ac,Deploying your Llama Model via vLLM using SageMaker Endpoint,2024-09-12,retrieve the LLM model from your S3 bucket not covered in AWS’s existing documentation. The previous example assumes...


## 2.3 Embeddings
To search the chunks, each chunk is turned into a vector once with `all-MiniLM-L6-v2`, the baseline model of the toolbox.

The vectors are saved in `RAG_DIR` and reloaded at later runs. The file name contains the model name, so another embedding model would be embedded again and saved to its own file, instead of reusing these vectors.

In [21]:
from anatoolbox.corpus import configure_embedder, ensure_embeddings, load_embeddings, save_embeddings
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = "all-MiniLM-L6-v2"

def use_embedding_model(name):
  model = SentenceTransformer(name)
  configure_embedder(lambda texts: model.encode(texts, batch_size=64, normalize_embeddings=True,
                                                 show_progress_bar=len(texts) > 1000))

def embed(chunk_result, model_name=None):
  corpus = get_corpus(chunk_result["corpus"])
  model_name = model_name or EMBEDDING_MODEL
  model_slug = re.sub(r"\W+", "_", model_name)
  cache = RAG_DIR / f"embeddings_{corpus.name}_{model_slug}.npy"
  started = time.perf_counter()
  if cache.exists():
    load_embeddings(corpus, cache)
    print(f"Embeddings loaded: {len(corpus):,} vectors ({cache.name})")
  else:
    ensure_embeddings(corpus)
    save_embeddings(corpus, cache)
    print(f"Embeddings computed and saved: {len(corpus):,} chunks in {time.perf_counter() - started:.0f} s ({cache.name})")
  return corpus, cache

use_embedding_model(EMBEDDING_MODEL)
chunk_corpus, EMB_PATH = embed(chunks)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings loaded: 12,720 vectors (embeddings_core_articles_size200_overlap40_title_all_MiniLM_L6_v2.npy)


## 2.4 Retrieval strategies
Three strategies are available through `anatoolbox`:

| Strategy | How it ranks the chunks | Expected strength |
|---|---|---|
| **sparse** | chunks that contain the words of the question, rare words weighing more | exact names, versions and numbers |
| **dense** | cosine similarity between the embeddings of the question and of the chunks | paraphrases and questions worded differently from the article |
| **hybrid** | reciprocal rank fusion: the two rankings are merged, a chunk ranked high by either strategy rises | combines both, without having to weigh their scores |

To see how the strategies behave, they are run on one question of each type from the test set, drawn at random (fixed seed) from five different articles. For each strategy, a table shows the top-5 articles and whether the source article of the question is among them.

This is a first look on five questions, not a measurement: the strategies are measured on the whole test set in the evaluation section.

In [22]:
def search(question, strategy="hybrid", size=30, **args):
    # Retrieve the `size` best chunks for `question` with the given strategy (sparse, dense or hybrid)
    return retrieve.run({"query": question, "input": chunks, "strategy": strategy, "size": size, **args},
                        context=ctx)

# One example question per type, drawn at random (fixed seed) from different source articles
EXAMPLES, used_types, used_articles = [], set(), set()
for q in pd.DataFrame(EVAL_SET).sample(frac=1, random_state=SEED).to_dict("records"):
    if q["type"] not in used_types and not set(q["source_ids"]) & used_articles:
        EXAMPLES.append(q)
        used_types.add(q["type"])
        used_articles.update(q["source_ids"])

def try_strategy(strategy):
    # For each example question: is the source article in the top-5 articles retrieved with `strategy`?
    rows = []
    for q in EXAMPLES:
        top = search(q["question"], strategy=strategy, size=10)["passages"]
        top_articles = list(dict.fromkeys(p["source_id"] for p in top))[:5]   # top-5 distinct articles
        rank = next((i + 1 for i, a in enumerate(top_articles) if a in q["source_ids"]), None)
        rows.append({"type": q["type"], "question": q["question"],
                     "source in top 5": f"yes (rank {rank})" if rank else "no",
                     "top article": top[0]["title"][:70]})
    return pd.DataFrame(rows)

### 2.4.1 Sparse search
Sparse search ranks the chunks that contains the words of the question with rare words weighing more. It should be strong on exact names, versions and numbers, and weak when the article uses other words than the question.

In [23]:
display(try_strategy("sparse"))

,type,question,source in top 5,top article
0,factual,Am I allowed to use the Llama name for my product?,no,"Discussion of Join the Gemma 4 Challenge: $ 3,000 prize pool for TEN w"
1,temporal,When did Google release Gemma 3?,no,"Welcome Gemma 3: Google's all new multimodal, multilingual, long conte"
2,relational,How is CrowdStrike using Nemotron Nano?,yes (rank 2),"CrowdStrike, Uber, Zoom Among Industry Pioneers Building Smarter Agent"
3,comparative,Which smaller Jais model outperformed a larger Jais model on OALL v2?,yes (rank 1),The Open Arabic LLM Leaderboard 2
4,analytical,What is the main performance advantage of DiffusionGemma?,yes (rank 1),"DiffusionGemma: Open-Weight Text at 1,000 Tokens/Sec"


Sparse search finds the source article in the top 5 for 3 of the 5 questions: rank 1 for the comparative (Jais) and analytical (DiffusionGemma) questions, and rank 2 for the relational question (CrowdStrike and Nemotron Nano). It misses the source for the factual question about the Llama name: the top article is a discussion of a Gemma 4 challenge. It also misses the source for the temporal question about Gemma 3, although its top article is about the Gemma 3 release (“Welcome Gemma 3…”), which suggests that several articles of the corpus might cover the same event.

### 2.4.2 Dense search
Dense search ranks the chunks by the cosine similarity between their embedding and the embedding of the question. It should find chunks worded differently from the question but may confuse close names.

In [24]:
display(try_strategy("dense"))

,type,question,source in top 5,top article
0,factual,Am I allowed to use the Llama name for my product?,yes (rank 1),meta-llama/Llama-3.2-3B-Instruct · Hugging Face
1,temporal,When did Google release Gemma 3?,yes (rank 3),Google Gemma 4: Complete Guide
2,relational,How is CrowdStrike using Nemotron Nano?,no,5 Big Moves In CrowdStrike-Nvidia AI Partnership
3,comparative,Which smaller Jais model outperformed a larger Jais model on OALL v2?,yes (rank 1),The Open Arabic LLM Leaderboard 2
4,analytical,What is the main performance advantage of DiffusionGemma?,yes (rank 1),"DiffusionGemma: Open-Weight Text at 1,000 Tokens/Sec"


Dense search finds the source article in the top 5 for 4 of the 5 questions: rank 1 for the factual (Llama name), comparative (Jais) and analytical (DiffusionGemma) questions, and rank 3 for the temporal question about Gemma 3. It misses the source for the relational question: the top article is about the CrowdStrike-Nvidia partnership, but it is not the source article. Compared with sparse search, dense search finds the two sources that sparse search missed (Llama, Gemma 3), but loses the CrowdStrike one.

### 2.4.3 Hybrid search
Hybrid search merges the sparse and dense rankings with reciprocal rank fusion: a chunk ranked high by either strategy rises, without having to weight their scores. It should combines the strengths of both.

In [25]:
display(try_strategy("hybrid"))

,type,question,source in top 5,top article
0,factual,Am I allowed to use the Llama name for my product?,yes (rank 1),meta-llama/Llama-3.2-3B-Instruct · Hugging Face
1,temporal,When did Google release Gemma 3?,yes (rank 2),Google Gemma 4: Complete Guide
2,relational,How is CrowdStrike using Nemotron Nano?,yes (rank 3),"CrowdStrike, Uber, Zoom Among Industry Pioneers Building Smarter Agent"
3,comparative,Which smaller Jais model outperformed a larger Jais model on OALL v2?,yes (rank 1),The Open Arabic LLM Leaderboard 2
4,analytical,What is the main performance advantage of DiffusionGemma?,yes (rank 1),"DiffusionGemma: Open-Weight Text at 1,000 Tokens/Sec"


Hybrid search finds the source article in the top 5 for all 5 questions: rank 1 for the factual (Llama name), comparative (Jais) and analytical (DiffusionGemma) questions, rank 2 for the temporal question about Gemma 3, and rank 3 for the relational question about CrowdStrike. It combines the results of both strategies: it keeps the sources found by dense search (Llama, Gemma 3) and the one found only by sparse search (CrowdStrike), although the last two are lower in the ranking.

### 2.4.4 Comparison and choice
Sparse search finds the source for three questions and dense search for four, but not the same ones. Sparse search misses the factual and temporal questions, which dense search finds. Dense search misses the relational question, which sparse search finds at rank 2. Hybrid search is the only strategy that finds the source for all five questions. Its ranks are lower than the best single strategy for two questions: Gemma 3 at rank 2 instead of 3 with dense is better, but CrowdStrike is at rank 3 instead of 2 with sparse.

**Verdict**: hybrid search is used for generation, since it is the only strategy that finds the source of all five questions, including the ones missed by sparse search and by dense search. This choice rests on five questions only: it is provisional, and `RETRIEVAL_ARGS` is updated if the evaluation on the whole test set shows that another strategy performs better.

In [26]:
RETRIEVAL_ARGS = {"strategy": "hybrid", "size": 30}

## 2.5 Grounded generation
The answer is generated by `synthesize_answer`, the generation tool of the toolbox. Its prompt has three parts: a system prompt with the rules (printed below), the question, and the context: at most 6 retrieved passages (2 per article, 1,200 characters each), labelled `[S1]…[Sn]` with their title and publication date.
- **Grounded mode**: the model may only use the passages, and must say when they do not answer the question.
- **Citations**: every sentence must cite its sources; the tool checks them (citations to unknown sources, share of sentences with a citation).
- **Dates**: the model is told to use the publication dates when the question concerns time.

In [27]:
ANSWER_ARGS = {"mode": "grounded", "max_passages": 6, "max_per_source": 2,
               "max_chars_per_passage": 1200, "max_tokens": 400}

def answer(question, passages=None, **args):
    # Answer `question` from retrieved passages (retrieved with RETRIEVAL_ARGS if none are given)
    passages = passages or search(question, **RETRIEVAL_ARGS)
    return synthesize.run({"question": question, "input": passages, **ANSWER_ARGS, **args}, context=ctx)

print(synthesize.system_prompt({"mode": "grounded"}))

You answer questions using numbered sources from a document collection.

Rules:
- Base every statement on the sources. If they do not contain enough information to answer, say so plainly and answer only what they support.
- Cite with the source labels in square brackets right after the sentence they support, for example [S1] or [S2][S4]. Do not collect citations at the end of the answer.
- Use only the labels you are given. Never invent sources, URLs, names, figures, or dates.
- When the question concerns time — what is new, what changed, when something happened — use the sources' publication dates and mention them.
- Where sources disagree, say so and cite each side.
- Write concise markdown. Do not end with a list of sources; the application adds one.


The same five example question are answered with hybrid search. Each answer is shown with the reference answer of the test set, and the table gives the cited sources with their dates and the citation checks.

In [28]:
results = []
for q in EXAMPLES:
    started = time.perf_counter()
    result = answer(q["question"])
    sources = {s["label"]: s for s in result["sources"]}
    display(Markdown(f"**[{q['type']}] {q['question']}**  \n{result['answer_markdown']}  \n"
                     f"*Reference answer:* {q['reference_answer']}"))
    results.append({"type": q["type"], "seconds": round(time.perf_counter() - started, 1),
                    "cited sources (date)": ", ".join(f"{label} ({sources[label]['date']})"
                                                      for label in result["cited"] if label in sources),
                    "unknown citations": len(result["unknown_citations"]),
                    "citation coverage": result["citation_coverage"],
                    "source article retrieved": any(s.get("source_id") in q["source_ids"] for s in result["sources"])})
display(pd.DataFrame(results))

**[factual] Am I allowed to use the Llama name for my product?**  
No, you are not allowed to use the "Llama" name for your product without permission or in a way that violates Meta's licensing terms. According to [S1](https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct), Meta grants a license to use the name "Llama" only "as required to comply with the last sentence of Section 1.b.i" — specifically, to include "Llama" at the beginning of any AI model name if you use Llama Materials to create, train, or improve an AI model. This implies that using "Llama" in a product name outside of such specific, authorized contexts is not permitted.

Additionally, [S2](https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct) explicitly states that "nor Licensee may use any name or mark owned by or associated with the other or any of its affiliates, except as required for reasonable and customary use in describing and redistributing the Llama Materials or as set forth in this Section 5 (a)." This reinforces that the use of the "Llama" name is restricted and must be limited to specific, authorized uses.

The RamaLama project, while inspired by "Llama" and using LLaMA.cpp, does not use the name "Llama" in a product name but instead uses "RamaLama" as a creative name derived from a children's song, indicating that a name incorporating "Llama" is not automatically permissible or endorsed [S3](https://thenewstack.io/ramalama-project-brings-containers-and-ai-together/). This further supports that the use of "Llama" in a product name is not freely available.

Therefore, using the "Llama" name for your product would likely violate Meta's brand and licensing guidelines unless explicitly authorized under the terms of the Llama 3.2 or Llama 3.3 Community License Agreement [S1](https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct)[S2](https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct).  
*Reference answer:* Only in a limited way: Meta grants no trademark licence except a licence to use "Llama" as required by Section 1.b.i of the Llama licence, and all goodwill from its use goes to Meta.

**[temporal] When did Google release Gemma 3?**  
Google released Gemma 3 on March 12, 2025 [S4](https://huggingface.co/blog/gemma3?utm_source=deepmind.google&utm_medium=referral&utm_campaign=gdm&utm_content=). This is supported by the source "Welcome Gemma 3: Google's all new multimodal, multilingual, long context open LLM" which states that Google released Gemma 3 on that date [S4](https://huggingface.co/blog/gemma3?utm_source=deepmind.google&utm_medium=referral&utm_campaign=gdm&utm_content=). Additionally, another source confirms the release of Gemma 3 in March 2025, noting it as one of Google Cloud’s new AI products launched that month [S2](https://www.crn.com/news/ai/2025/google-cloud-s-10-new-bold-ai-products-gemini-robotics-ai-coach-and-gemma-3). The specific date of March 12, 2025, is directly cited in the Hugging Face blog post [S4](https://huggingface.co/blog/gemma3?utm_source=deepmind.google&utm_medium=referral&utm_campaign=gdm&utm_content=).  
*Reference answer:* In March 2025.

**[relational] How is CrowdStrike using Nemotron Nano?**  
CrowdStrike is fine-tuning the NVIDIA Nemotron Nano model for cyber defense, using it to generate investigation queries within the Falcon LogScale platform. Internal testing has achieved 96% accuracy in generating these queries, providing a natural-language interface that enhances the efficiency of agent investigations [S5](https://blogs.nvidia.com/blog/open-secure-ai-alliance-contributions/). Additionally, CrowdStrike has published research showing that a specialized Nemotron Nano reasoning model outperforms larger models in Security Operations Center (SOC) detection triage, with calibrated logit-based confidence scores enabling measurable, tunable, and auditable autonomous security decisions [S5](https://blogs.nvidia.com/blog/open-secure-ai-alliance-contributions/). CrowdStrike is also leveraging the NeMo Data Designer to fine-tune Nemotron models with data from its own tools, allowing for the creation of domain-specific, threat-hunting agents that can be deployed at the edge and scaled across enterprise environments [S4](https://www.crn.com/news/ai/2025/5-big-moves-in-crowdstrike-nvidia-ai-partnership)[S2](https://www.crn.com/news/ai/2025/5-big-moves-in-crowdstrike-nvidia-ai-partnership). This integration supports the development of smarter, composable AI agents that operate effectively at the edge, enhancing cybersecurity response speed and scalability [S2](https://www.crn.com/news/ai/2025/5-big-moves-in-crowdstrike-nvidia-ai-partnership).  
*Reference answer:* CrowdStrike is fine-tuning it for cyber defense: internal testing reached 96% accuracy in generating investigation queries in Falcon LogScale.

**[comparative] Which smaller Jais model outperformed a larger Jais model on OALL v2?**  
The smaller Jais model that outperformed a larger Jais model on OALL v2 is the inceptionai/jais-family-30b-8k model, which surpasses the larger inceptionai/jais-adapted-70b model on OALL v2 [S1](https://huggingface.co/blog/leaderboard-arabic-v2). This finding is supported by the analysis in Figure 6 of the Open Arabic LLM Leaderboard 2, which shows that while larger models generally achieve higher average scores, this exception occurs in the Jais family [S1](https://huggingface.co/blog/leaderboard-arabic-v2).  
*Reference answer:* inceptionai/jais-family-30b-8k outperformed inceptionai/jais-adapted-70b.

**[analytical] What is the main performance advantage of DiffusionGemma?**  
The main performance advantage of DiffusionGemma is its significantly reduced per-request latency due to parallel text generation, achieved by denoising up to 256 tokens at once rather than generating one token at a time [S5](https://www.thefastmode.com/technology-solutions/48931-google-deepmind-unveils-google-deepmind-diffusiongemma-a-new-era-of-fast-text-generation)[S6](https://www.thefastmode.com/technology-solutions/48931-google-deepmind-unveils-google-deepmind-diffusiongemma-a-new-era-of-fast-text-generation). This parallelism shifts the computational workload from memory-bound to compute-bound, leveraging the strengths of GPU architectures such as NVIDIA Tensor Cores and CUDA, resulting in a speedup that makes it particularly effective for single-user, interactive workloads like real-time chat or on-device assistants [S6](https://www.thefastmode.com/technology-solutions/48931-google-deepmind-unveils-google-deepmind-diffusiongemma-a-new-era-of-fast-text-generation). Compared to standard autoregressive models, DiffusionGemma is described as being 4x faster on H100 hardware, delivering 1,000 tokens per second [S2](https://dev.to/max_quimby/diffusiongemma-open-weight-text-at-1000-tokenssec-16o5). This advantage is especially notable in latency-sensitive scenarios where responsiveness matches human thinking and iteration patterns [S6](https://www.thefastmode.com/technology-solutions/48931-google-deepmind-unveils-google-deepmind-diffusiongemma-a-new-era-of-fast-text-generation). While this design excels in single-user settings, it does not provide a significant benefit in high-throughput, multi-user cloud environments where batched, shared GPU usage is more efficient [S2](https://dev.to/max_quimby/diffusiongemma-open-weight-text-at-1000-tokenssec-16o5).  
*Reference answer:* Lower latency for individual requests rather than higher overall throughput.

,type,seconds,cited sources (date),unknown citations,citation coverage,source article retrieved
0,factual,5.5,"S1 (2024-09-18), S2 (2024-09-18), S3 (2025-03-18)",0,0.5,True
1,temporal,2.6,"S4 (2025-03-12), S2 (2025-03-14)",0,1.0,True
2,relational,3.5,"S5 (2026-08-04), S4 (2025-10-28), S2 (2025-10-28)",0,0.8,True
3,comparative,2.3,S1 (2025-04-08),0,1.0,True
4,analytical,3.7,"S5 (2026-06-11), S6 (2026-06-11), S2 (2026-06-28)",0,1.0,True


For the five questions, the source article is among the retrieved passages, and no answer contains an unknown citation: every citation refers to a passage given to the model. Citation coverage ranges from 0.5 (factual) to 1.0, which is reached by the temporal, comparative and analytical answers; the relational answer reaches 0.8. The answers cite between one and three sources, with their publication dates, from 2024-09-18 to 2026-08-04.

## 2.6 Limitations
The examples of this section show several limitations of the baseline (hybrid search) that could be addressed by an enhancement:
- **Close names**: for the question about Gemma 3, dense and hybrid search rank an artical about Gemma 4 first, before the source article; a re-ranking step could favour the exact model name
- **General wording**: sparse search, one of the two parts of the hybrid search, misses the question made of common words (the Llama name), and its top article is unrelated (a Gemma 4 challenge, not Llama); hybrid search still finds the source thanks to dense search, but expanding the query with related terms could make sparse search more reliable.
- **Uncited parts**: the citation coverage of the factual and relation answers is below 1.0, so part of these answers is not linked to a source; asking the model to cite every sentence could make the answers easier to check.

## 2.7 RAG outputs
The settings of the baseline are saved in `rag_settings.json`, so that the knowledge-graph integration, the evaluation and the enhancement use exactly the same system. The embeddings are already saved in `RAG_DIR`.



In [29]:
RAG_SETTINGS = {
    "corpus": {"file": CORE_CSV.name, "articles": rag_articles["records"], "filter": "Stage 1 om_tier == 'core'"},
    "chunks": CHUNK_ARGS,
    "embedding_model": EMBEDDING_MODEL,
    "embeddings_file": EMB_PATH.name,
    "retrieval": RETRIEVAL_ARGS,
    "generation": ANSWER_ARGS,
    "rag_model": {"model": RAG_MODEL, "server": "Ollama, OpenAI-compatible endpoint http://localhost:11434/v1"},
    "test_set": {"file": "qa_reviewed.json", "questions": len(EVAL_SET)},
}
(RAG_DIR / "rag_settings.json").write_text(json.dumps(RAG_SETTINGS, indent=2), encoding="utf-8")
print(json.dumps(RAG_SETTINGS, indent=2))

{
  "corpus": {
    "file": "core_articles.csv",
    "articles": 1210,
    "filter": "Stage 1 om_tier == 'core'"
  },
  "chunks": {
    "size": 200,
    "overlap": 40,
    "contextualize": "title"
  },
  "embedding_model": "all-MiniLM-L6-v2",
  "embeddings_file": "embeddings_core_articles_size200_overlap40_title_all_MiniLM_L6_v2.npy",
  "retrieval": {
    "strategy": "hybrid",
    "size": 30
  },
  "generation": {
    "mode": "grounded",
    "max_passages": 6,
    "max_per_source": 2,
    "max_chars_per_passage": 1200,
    "max_tokens": 400
  },
  "rag_model": {
    "model": "qwen3:4b-instruct",
    "server": "Ollama, OpenAI-compatible endpoint http://localhost:11434/v1"
  },
  "test_set": {
    "file": "qa_reviewed.json",
    "questions": 213
  }
}


## 2.7 Saved outputs
| File | Content |
|---|---|
| `core_articles.csv` | the 1,210 core articles ingested by the RAG system (id = Stage 1 `doc_id`) |
| `embeddings_…all_MiniLM_L6_v2.npy` | the embeddings of the chunks, reloaded at later runs |
| `rag_settings.json` | the settings of the baseline, reused by the other work packages |

## AI assistance Disclosure
- **Q&A drafting**: Claude Opus 5.5 (Anthropic) drafted the questions and reference answers from the selected passages. The brief requires the Q&A pairs to be generated by a different LLM than the one used in the RAG. A large proprietary model was chosen for this step because it writes more natural and precise questions than a small open model. All drafted questions were then reviewed manually by the team.
- **Coding assistant**: Claude was used to write and debug code and to suggest corrections in the notebook
- **Writing**: Claude helped shorten and rephrase the markdown texts. All interpretations are based on the outputs of the notebook and were written and validated by the team member.



In [ ]:
# Push the notebook and the Stage 3 outputs to GitHub
# Publishing step (not part of the analysis): needs a GITHUB_TOKEN in the Colab secrets
import shutil, subprocess
from google.colab import userdata, _message

REPO_URL = "github.com/jihyeon26/computational-language-technology-hs26.git"
BRANCH = "main"
REPO_ARTIFACTS = REPO / "artifacts" / "stage3_wp1"
REPO_NOTEBOOK = REPO / "stage3" / "work_packages" / "wp1" / "stage3_wp1_qa_rag.ipynb"
GIT_NAME = "Jenniferspr"
GIT_EMAIL = "173487990+JenniferSpr@users.noreply.github.com"
SMALL_FILES = [FINAL_JSON, REVIEWED_PATH, DRAFT_PATH, PASSAGES_PATH, EXTRA_PATH, RAG_DIR / "rag_settings.json"]

token = userdata.get("GITHUB_TOKEN")
remote = f"https://x-access-token:{token}@{REPO_URL}"

def git(*args, check=True):
    """Run a git command in the repository without ever printing the token."""
    r = subprocess.run(["git", "-C", str(REPO), *args], capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(token, "***")
    if out.strip():
        print(out.strip())
    if check and r.returncode != 0:
        raise RuntimeError(f"git {args[0]} failed (see message above)")
    return r

# 1. Latest version of the repository, commit identity
git("config", "user.name", GIT_NAME)
git("config", "user.email", GIT_EMAIL)
git("rebase", "--abort", check=False)      # clean up an interrupted rebase, if any
git("checkout", BRANCH)
git("pull", "--rebase", remote, BRANCH)

# 2. This notebook as it is now in Colab, with its outputs
notebook = _message.blocking_request("get_ipynb", timeout_sec=60)["ipynb"]
REPO_NOTEBOOK.parent.mkdir(parents=True, exist_ok=True)
REPO_NOTEBOOK.write_text(json.dumps(notebook, indent=1, ensure_ascii=False), encoding="utf-8")

# 3. Small outputs (the test set and the RAG settings)
REPO_ARTIFACTS.mkdir(parents=True, exist_ok=True)
for path in SMALL_FILES:
    shutil.copy2(path, REPO_ARTIFACTS / path.name)

# 4. .gitignore excludes artifacts/ and *.csv: add an exception for this folder (once)
gitignore = REPO / ".gitignore"
rule = "!artifacts/stage3_wp1/\n!artifacts/stage3_wp1/**\n"
if "!artifacts/stage3_wp1/" not in gitignore.read_text():
    gitignore.write_text(gitignore.read_text().rstrip("\n")
                         + "\n\n# Exception: Stage 3 WP1 Q&A test set and RAG settings are versioned\n" + rule)

# 5. README: what is here and where the large files are
test_set = json.loads(FINAL_JSON.read_text(encoding="utf-8"))
types = pd.Series([q["type"] for q in test_set]).value_counts().to_dict()
(REPO_ARTIFACTS / "README.md").write_text(
    "# Stage 3 - WP1: Q&A test set and text-only RAG baseline (Open Models)\n\n"
    f"Notebook: `{REPO_NOTEBOOK.relative_to(REPO)}`\n\n"
    "| file | content |\n|---|---|\n"
    f"| `{FINAL_JSON.name}` | **final test set**: {len(test_set)} questions ({', '.join(f'{t} {n}' for t, n in types.items())}). "
    "Fields: `id`, `type`, `question`, `reference_answer`, `source_ids` (Stage 1 `doc_id` of the source article), "
    "`date`, `passage_id`, `quarter`, `topic` |\n"
    f"| `{REVIEWED_PATH.name}` | all drafted questions after the manual review (keep / edit / drop), with notes and reviewer |\n"
    f"| `{DRAFT_PATH.name}` | the drafted questions before the review |\n"
    f"| `{PASSAGES_PATH.name}` | the 150 passages the questions were drafted from |\n"
    f"| `{EXTRA_PATH.name}` | the 15 extra passages with an explicit date (temporal questions) |\n"
    "| `rag_settings.json` | settings of the RAG baseline: corpus, chunks, embedding model, retrieval, generation, RAG model |\n\n"
    "## RAG baseline\n"
    f"- Corpus: {rag_articles['records']:,} core articles (`om_tier == 'core'`), `id` = Stage 1 `doc_id`\n"
    f"- Chunks: {CHUNK_ARGS}; embeddings: `{EMBEDDING_MODEL}`\n"
    f"- Retrieval: {RETRIEVAL_ARGS}, provisional (chosen on 5 examples, to be confirmed by the evaluation)\n"
    f"- Generation: {ANSWER_ARGS}\n"
    f"- RAG model: `{RAG_MODEL}` with Ollama (local, no API key). "
    "The brief requires a **different** model for the evaluation (judge).\n\n"
    "## Large files (shared Google Drive, not on GitHub)\n"
    "`HSLU/CLT/extraction_outputs/stage3/rag/`: `core_articles.csv` (the ingested corpus) and the chunk embeddings "
    f"`{EMB_PATH.name}`. Copy them into your `RAG_DIR` to reload them instead of recomputing them.\n",
    encoding="utf-8")

# 6. Commit and push (only if something changed)
git("add", ".gitignore", str(REPO_ARTIFACTS.relative_to(REPO)), str(REPO_NOTEBOOK.relative_to(REPO)))
if git("diff", "--cached", "--quiet", check=False).returncode == 0:
    print("Nothing new to push")
else:
    git("commit", "-m", "Add Stage 3 WP1 notebook, Q&A test set and RAG baseline settings")
    git("push", remote, f"HEAD:{BRANCH}")
    print(f"Pushed: {REPO_NOTEBOOK.relative_to(REPO)} and {REPO_ARTIFACTS.relative_to(REPO)}/")

fatal: No rebase in progress?
Your branch is up to date with 'origin/main'.
Already on 'main'
